## 연습 1: 원본 예약 데이터 수집 (Bronze Layer)

**Bronze 계층**은 원본 데이터를 정확히 소스 시스템에서 수신한 대로 저장합니다. 프로덕션 파이프라인에서는 데이터가 Property Management System(PMS)에서 도착할 것입니다. 이 실습에서는 5개의 GlobStay 속성 전체의 예약을 나타내는 샘플 데이터 세트가 제공됩니다.

데이터 문제는 의도적입니다. Silver 계층에서 해결할 실제 조건을 반영합니다.

### ✅ 제공됨: Unity Catalog 구조 및 샘플 데이터 생성

아래 두 셀을 실행하여:
- `bronze`, `silver`, `gold` 스키마를 포함한 `hospitality_lab` 카탈로그 생성
- 의도적인 품질 문제가 포함된 샘플 원본 예약 DataFrame 생성

출력을 주의 깊게 검사하십시오. 연습 2를 시작하기 전에 데이터 문제를 이해해야 합니다.

In [ ]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

In [ ]:
# ✅ Run this cell — creates the Unity Catalog structure

catalog = "hospitality_lab"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog} MANAGED LOCATION '{storage_root}' COMMENT 'GlobStay hospitality analytics platform'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze COMMENT 'Raw ingested booking data'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver COMMENT 'Cleaned and validated booking data'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold   COMMENT 'Analytics-ready aggregated data'")

print(f"✅ Catalog '{catalog}' and schemas bronze / silver / gold created.")